In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    raise ValueError("La variable DATABASE_URL est absente du fichier .env")

engine = create_engine(DATABASE_URL)

In [ ]:
%pip install python-dotenv

In [ ]:
query = """
SELECT *
FROM orders
"""

orders_sample = pd.read_sql(query, engine)

orders_sample

In [ ]:
orders_sample.info()

In [ ]:
orders_sample.isna().sum()

In [ ]:
orders_sample["status"].value_counts()

In [ ]:
orders_count = pd.read_sql(
    "SELECT COUNT(*) AS total_orders FROM orders",
    engine
)

orders_count

In [ ]:
query = """
SELECT
    status,
    COUNT(*) AS nombre_commandes,
    SUM(total_amount) AS chiffre_affaires
FROM orders
GROUP BY status
ORDER BY nombre_commandes DESC
"""

orders_status = pd.read_sql(query, engine)

orders_status

In [ ]:
query = """
SELECT
    COUNT(*) AS nombre_commandes,
    SUM(total_amount) AS chiffre_affaires
FROM orders
WHERE status <> 'cancelled'
"""

sales_summary = pd.read_sql(query, engine)
#pd.options.display.float_format = '{:,.2f}'.format
sales_summary.style.format({
    'chiffre_affaires': '{:,.0f}',
    'panier_moyen': '{:,.2f}'
})
sales_summary


In [ ]:
# Chiffre d'affaires par année
# Les commandes annulées sont exclues

query = """
SELECT
    YEAR(order_date) AS annee,
    COUNT(*) AS nombre_commandes,
    SUM(total_amount) AS chiffre_affaires
FROM orders
WHERE status <> 'cancelled'
GROUP BY YEAR(order_date)
ORDER BY annee
"""

sales_by_year = pd.read_sql(query, engine)

sales_by_year

In [ ]:
orders_sample.shape # return nbrs rows-columns

In [ ]:
orders_sample.columns #return list of columns

In [ ]:
orders_sample.dtypes #return type of data of every columns

In [ ]:
orders_sample.info()

In [ ]:
orders_sample.describe()

In [ ]:
orders_sample["status"].value_counts()

In [ ]:
orders_sample["status"].nunique()

In [ ]:
orders_sample["status"].unique()

In [ ]:
orders_sample.isna().sum()

In [ ]:
orders_sample.isnull().sum() #utilisés pour détecter les valeurs manquantes

In [ ]:
orders_sample.duplicated().sum()

In [ ]:
orders_sample.duplicated()

In [ ]:
orders_sample["order_id"].duplicated().sum()

In [ ]:
orders_sample.sort_values("total_amount", ascending=False).head(10)#récupérer les 10 grands total 
#si on utilise true on va récupérer les petits

In [ ]:
orders_high_value = orders_sample[orders_sample["total_amount"] > 40000]
orders_high_value.head(10)

In [ ]:
orders_high_value_completed = orders_sample[
    (orders_sample["total_amount"] > 40000) &
    (orders_sample["status"] == "completed")
]
orders_high_value_completed.head(100)

In [ ]:
orders_completed_or_delivered = orders_sample[
    (orders_sample["status"] == "completed") |
    (orders_sample["status"] == "delivered")
]
orders_completed_or_delivered["status"].value_counts()

In [ ]:
orders_completed_or_delivered = orders_sample[
    orders_sample["status"].isin(["completed", "delivered"])
]

orders_completed_or_delivered["status"].value_counts()

In [ ]:
orders_between_10000_20000 = orders_sample[
    orders_sample["total_amount"].between(10000, 20000)
]

orders_between_10000_20000[["order_id", "total_amount", "status"]].head(10)

In [ ]:
orders_completed = orders_between_10000_20000.loc[
    orders_between_10000_20000["status"] == "completed"
]

orders_completed.head(10)

In [ ]:
orders_completed = orders_between_10000_20000.loc[
    orders_between_10000_20000["status"] == "completed",
    ["order_id", "shipping_city", "status", "total_amount"]
]

orders_completed.head(10)

In [ ]:
orders_delivered=orders_between_10000_20000.loc[
    orders_between_10000_20000["status"]=='delivered',
    ["order_id","shipping_city","total_amount"]
].head(10)

orders_delivered

In [ ]:
orders_sample.iloc[0:5].shape

In [ ]:
orders_sample.iloc[1:6, [3, 10]]

In [ ]:
orders_sample.groupby("status").size()

In [ ]:
orders_sample.groupby("status")["total_amount"].mean()

In [ ]:
orders_sample.groupby("status")["total_amount"].agg(
    ["count", "sum", "mean"]
)

In [ ]:
orders_sample.groupby("status")["total_amount"].min()

In [ ]:
orders_sample.groupby("status")["total_amount"].agg(
    ["min", "max", "mean", "sum"]
)

In [ ]:
orders_sample.groupby("status")["total_amount"].agg(
    montant_min="min",
    montant_max="max",
    montant_moyen="mean",
    chiffre_affaires="sum"
)

In [ ]:
resultat = orders_sample.groupby("status")["total_amount"].agg(
    montant_min="min",
    montant_max="max",
    montant_moyen="mean",
    chiffre_affaires="sum"
)

#resultat = resultat.round(2)
pd.set_option("display.float_format", "{:,.2f}".format)


resultat

In [ ]:
orders_sample["annee"] = orders_sample["order_date"].dt.year

orders_sample[["order_date", "annee"]].head()

In [ ]:
commandes_par_annee_statut = orders_sample.groupby(
    ["annee", "status"]
)["total_amount"].sum()

commandes_par_annee_statut

In [ ]:
ca_par_annee_statut = orders_sample.groupby(
    ["annee", "status"]
)["total_amount"].sum()

ca_par_annee_statut = ca_par_annee_statut.reset_index()

ca_par_annee_statut

In [ ]:
ca_par_annee_statut = ca_par_annee_statut.rename(
    columns={"total_amount": "chiffre_affaires"}
)

ca_par_annee_statut

In [ ]:
ca_par_annee_statut = ca_par_annee_statut.sort_values(
    "chiffre_affaires",
    ascending=False
)

ca_par_annee_statut

In [ ]:
top_5_ca = ca_par_annee_statut.head(5)

top_5_ca

In [ ]:
orders_sample.tail(10)

In [ ]:
orders_sample[["order_id","status","total_amount"]].head(5)

In [ ]:
order_comm = orders_sample.loc[
    (orders_sample["status"] =='delivered')&(orders_sample["total_amount"]>30000),
    ["order_id", "shipping_city", "total_amount"]
]

order_comm

In [ ]:
res=orders_sample.query(
    'status =="completed" and total_amount>40000'
)[["order_id", "status","total_amount"]]

res=res.reset_index(drop=True)

res.head()

In [ ]:
resultat = orders_sample.query(
    'status == "delivered" and total_amount > 30000'
).reset_index(drop=True)

resultat.head()

In [ ]:
resultat = orders_sample.query(
    'status == "delivered" and total_amount > 30000'
).copy()

resultat["test"] = "OK"

resultat.head()

In [ ]:
"test" in orders_sample.columns

In [ ]:
orders_sample["customer_id"].astype(str).dtype

In [ ]:
orders_sample["order_date"].dtype

In [ ]:
date_texte = orders_sample["order_date"].astype(str)

print(date_texte.dtype)
print(orders_sample["order_date"].dtype)

In [ ]:
date_convertie = pd.to_datetime(date_texte)

print(date_convertie.dtype)
print(date_convertie.head())

In [ ]:
orders_sample["jour"] = orders_sample["order_date"].dt.day_name()

orders_sample[["order_date", "jour"]].head()

In [ ]:
orders_sample["nom_jour_fr"] = orders_sample["jour"].replace({
    "Monday": "Lundi",
    "Tuesday": "Mardi",
    "Wednesday": "Mercredi",
    "Thursday": "Jeudi",
    "Friday": "Vendredi",
    "Saturday": "Samedi",
    "Sunday": "Dimanche"
})

orders_sample[["order_date", "jour", "nom_jour_fr"]].head()

In [ ]:
commandes_par_jour = orders_sample["nom_jour_fr"].value_counts()

print(commandes_par_jour)

In [ ]:
pourcentage_par_jour = (
    orders_sample["nom_jour_fr"].value_counts(normalize=True) * 100
).round(2)

print(pourcentage_par_jour)

In [ ]:
orders_sample.isna().sum()

In [ ]:
orders_sample["order_id"].duplicated().sum()

In [ ]:
orders_sample.duplicated().sum()

In [ ]:
orders_nettoyees = orders_sample.drop_duplicates()

print("Avant :", orders_sample.shape)
print("Après :", orders_nettoyees.shape)

In [ ]:
orders_sample["order_id"].duplicated().sum()

In [ ]:
orders_sample.dropna().shape

In [ ]:
montants = pd.Series([100, None, 300, None, 500])

montants.fillna(0)

In [ ]:
montants_nettoyes = montants.fillna(0)

print(montants_nettoyes)

In [ ]:
import numpy as np

ventes = np.array([150, 300, 450, 600,900])

print(ventes)

In [ ]:
print(ventes.shape)
print(ventes.ndim)
print(ventes.dtype)

In [ ]:
print(np.sum(ventes))

In [ ]:
print(np.mean(ventes))

In [ ]:

import numpy as np

ventes = np.array([150, 300, 450, 600, 900])

ventes_avec_reduction = ventes * 0.1

print(ventes_avec_reduction)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
jours = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi"]
ventes = [150, 300, 450, 190, 500]

plt.plot(jours, ventes)
plt.show()

In [ ]:
print(orders_sample.columns.tolist())

In [ ]:
print(orders_sample.columns.tolist())

In [ ]:
orders_sample["order_date"] = pd.to_datetime(
    orders_sample["order_date"]
)

orders_sample["mois"] = orders_sample["order_date"].dt.month

In [ ]:

ventes_mensuelles = (
    orders_sample[orders_sample["status"] != "cancelled"]
    .groupby(["annee", "mois"], as_index=False)["total_amount"]
    .sum()
    .sort_values(["annee", "mois"])
)

print(ventes_mensuelles.head(12))


In [ ]:
ventes_2025 = (
    orders_sample[
        (orders_sample["status"] != "cancelled") &
        (orders_sample["annee"] == 2025)
    ]
    .groupby("mois", as_index=False)["total_amount"]
    .sum()
    .sort_values("mois")
)

plt.figure(figsize=(10, 5))

plt.bar(
    ventes_2025["mois"],
    ventes_2025["total_amount"]
)

plt.xlabel("Mois")
plt.ylabel("Total des ventes")
plt.title("Ventes mensuelles - 2025")
plt.xticks(range(1, 13))
plt.ticklabel_format(axis="y", style="plain")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Filtrer les commandes
df = orders_sample[
    (orders_sample["status"] != "cancelled") &
    (orders_sample["annee"] == 2025)
].copy()

# 2. Calculer les ventes par mois
ventes_2025 = df.groupby("mois")["total_amount"].sum()

# 3. Afficher les résultats
print(ventes_2025)


# 4. Tracer le graphique
plt.figure(figsize=(10, 5))
plt.bar(ventes_2025.index, ventes_2025.values, width=0.7)

plt.xlabel("Mois")
plt.ylabel("Total des ventes")
plt.title("Ventes mensuelles - 2025")
plt.xticks(range(1, 13))
plt.xlim(0.5, 12.5)
plt.ticklabel_format(axis="y", style="plain")
plt.tight_layout()
plt.show()

In [ ]:

ventes_2025 = (
    df.groupby("mois")["total_amount"]
    .sum()
    .reindex(range(1, 13), fill_value=0)
)

print(ventes_2025)
print("Nombre de mois :", len(ventes_2025))


In [ ]:
fichier = dossier_graphiques / "01_ca_mensuel_2025.png"

print("Image enregistrée :", fichier.exists())
print("Chemin du fichier :", fichier)

In [ ]:

plt.figure(figsize=(12, 6))

plt.bar(ventes_2025.index, ventes_2025.values, width=0.7)

# Ajouter le montant au-dessus de chaque barre
for mois, montant in zip(ventes_2025.index, ventes_2025.values):
    plt.text(mois, montant, f"{montant / 1_000_000:.1f} M",
             ha="center", va="bottom", fontsize=8)

plt.xlabel("Mois")
plt.ylabel("Ventes (DH)")
plt.title("Évolution des ventes mensuelles - 2025")
plt.xticks(range(1, 13))
plt.ticklabel_format(axis="y", style="plain")
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(
    dossier_graphiques / "ventes_mensuelles.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
# Afficher le dossier de travail actuel
print("Dossier actuel :", Path.cwd())

# Définir le chemin absolu du dossier des graphiques
dossier_graphiques = Path(
    r"C:\Users\ELHADDOUCHI Asmae\Documents\ecommerce_data_analytics\reports\graphiques"
)

# Créer le dossier s'il n'existe pas
dossier_graphiques.mkdir(parents=True, exist_ok=True)

print("Dossier des graphiques :", dossier_graphiques)
print("Le dossier existe :", dossier_graphiques.exists())

In [ ]:
#print(ventes_2025.index.tolist())
#print("Nombre de barres attendu :", len(ventes_2025))

In [ ]:
df_status= orders_sample[
    (orders_sample["annee"]==2025) &
    (orders_sample["status"]!="cancelled")
].copy()


print(df_status.shape)
print(df_status["status"].unique())

In [ ]:
ca_status=df_status.groupby("status")["total_amount"].sum()

#print(ca_status)

In [ ]:
plt.figure(figsize=(10, 5))

# Ajouter le montant au-dessus de chaque barre

for status, montant in zip(ca_status.index,ca_status .values):
    plt.text(status, montant, f"{montant} DH",
             ha="center", va="bottom", fontsize=8.5)

plt.bar(ca_status.index , ca_status.values , width=0.9 , color='green')

plt.xlabel("Status des commandes")
plt.ylabel("CA (DH)")
plt.title("chiffre d’affaires par statut")
plt.savefig(
    dossier_graphiques / "CA_Par_Status.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
shipping_region=orders_sample[
    (orders_sample["annee"]==2025)&
    (orders_sample["status"]!='cancelled')
].copy()

ca_regions=shipping_region.groupby("shipping_region")["total_amount"].sum().rename({
    "B├®ni Mellal-Kh├®nifra": "Beni Mellal - khnifra",
    "F├¿s-Mekn├¿s": "Fés Meknés",
    "Rabat-Sal├®-K├®nitra": "Rabat - salé - kénitra",
    "Tanger-T├®touan-Al Hoce├»ma": "Tanger - Tétouan - El Hocima",

})

print(ca_regions)

In [ ]:
#quelle région génère le plus de chiffre d'affaires en 2025,
#  en excluant les commandes annulées ?

plt.figure(figsize=(17,7))
#ca_regions = ca_regions.sort_values(ascending=True)
#graphique Bar 
plt.bar(ca_regions.index , ca_regions.values , width=0.7 , color='purple')
#afficher montant au dessous de chaque barre 
for i , montant in zip(ca_regions.index,ca_regions.values):
    plt.text(i,montant,f"{montant}DH"
    ,ha='center',va="bottom", fontsize=10 , color="purple")

#les titres 
plt.xlabel("Régions")
plt.ylabel("CA (DH)")
plt.title("le chiffre d'affaires total par région")
plt.ticklabel_format(axis="y", style="plain")
plt.savefig(
    dossier_graphiques / "Ca_Par_Region.png",
    dpi=300,
    bbox_inches="tight"
)
#affichage

plt.show()


In [ ]:
#la majorité des commandes ont-elles un montant faible, moyen ou élevé ?

df_montants = orders_sample[
    (orders_sample["annee"] == 2025) &
    (orders_sample["status"] != "cancelled")
].copy()

print(df_montants["total_amount"].describe())


In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(df_montants["total_amount"], bins=20)

plt.xlabel("Montant des commandes (DH)")
plt.ylabel("Nombre de commandes")
plt.title("Distribution des montants des commandes en 2025")

plt.show()

In [ ]:
commandes_intervalle = df_montants[
    df_montants["total_amount"].between(20000, 30000)
]

print(len(commandes_intervalle))

In [ ]:
df_region=orders_sample[
    (orders_sample["annee"]==2025 )&(orders_sample["status"]!="cancelled")

].copy()

nb_commande=orders_sample.groupby("shipping_region").size()

print(nb_commande)

In [ ]:
plt.figure(figsize=(18,8))

plt.bar(nb_commande.index , nb_commande.values , width=0.6 , color="pink")

plt.title("Nombre de commandes par région")
plt.xlabel("région")
plt.ylabel("nombre commande")
plt.xticks(rotation=45)
plt.savefig(
    dossier_graphiques / "Nombre_de_commandes_par_région.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
montant_moyen_region = df_region.groupby("shipping_region")["total_amount"].mean()

print(montant_moyen_region)

In [ ]:
#un graphique en barres qui compare le montant 
# moyen des commandes par région en 2025, hors commandes annulées.
plt.figure(figsize=(12, 7))

plt.bar(
    montant_moyen_region.index,
    montant_moyen_region.values,
    width=0.6
)

plt.title("Montant moyen des commandes par région en 2025")
plt.xlabel("Région")
plt.ylabel("Montant moyen (DH)")
plt.xticks(rotation=45)
for i, montant in enumerate(montant_moyen_region.values):
    plt.text(i, montant, f"{montant:,.0f} DH", ha="center")
plt.show()

In [ ]:

plt.figure(figsize=(10, 6))

plt.scatter(
    orders_sample["shipping_fee"],
    orders_sample["total_amount"],
    alpha=0.4
)

plt.title("Relation entre les frais de livraison et le montant des commandes")
plt.xlabel("Frais de livraison (DH)")
plt.ylabel("Montant total (DH)")

plt.show()


In [ ]:
montant_moyen_livraison = (
    orders_sample
    .groupby("shipping_fee")["total_amount"]
    .mean()
)

print(montant_moyen_livraison)

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    montant_moyen_livraison.index,
    montant_moyen_livraison.values,
    width=2
)

plt.title("Montant moyen selon les frais de livraison")
plt.xlabel("Frais de livraison (DH)")
plt.ylabel("Montant moyen (DH)")
plt.tight_layout()
plt.show()

In [ ]:
#Cette instruction compte le nombre de commandes pour chaque tarif de livraison.

orders_sample["shipping_fee"].value_counts().sort_index()

In [ ]:
orders_sample["shipping_fee"].value_counts().idxmax()

In [ ]:
frequence_livraison = orders_sample["shipping_fee"].value_counts()

print(frequence_livraison)
print("Tarif le plus fréquent :", frequence_livraison.idxmax(), "DH")
print("Nombre de commandes :", frequence_livraison.max())

In [ ]:

nb_livraison_gratuite = (orders_sample["shipping_fee"] == 0).sum()

pourcentage_livraison_gratuite = (
    nb_livraison_gratuite / len(orders_sample)
) * 100

print("Nombre de commandes :", nb_livraison_gratuite)
print("Pourcentage :", round(pourcentage_livraison_gratuite, 2), "%")


In [ ]:
#Objectif métier : identifier les tarifs les plus fréquents 
# et comparer leur importance.

frequence_livraison = orders_sample["shipping_fee"].value_counts().sort_index()
plt.figure(figsize=(9, 5))

plt.bar(frequence_livraison.index.astype(str),frequence_livraison.values , width=0.7)
plt.title("le nombre de commandes par tarif de livraison")
plt.xlabel("Frais de livraison (DH)")
plt.ylabel("Nombre de commandes")
plt.tight_layout()
plt.show()


In [ ]:
#Les commandes avec livraison gratuite génèrent-elles plus de chiffre d'affaires que celles avec livraison payante ?

ca_par_livraison=orders_sample.groupby("shipping_fee")["total_amount"].sum()
print(ca_par_livraison)

In [ ]:
#Calculer le montant moyen par tarif


montant_moyen_livraison = (
    orders_sample
    .groupby("shipping_fee")["total_amount"]
    .mean()
)

print(montant_moyen_livraison.round(2))


In [ ]:
#comparer le montant moyen d'une commande selon les frais de livraison
plt.figure(figsize=(10, 6))
plt.bar(montant_moyen_livraison.index.astype(str) , montant_moyen_livraison.values , width=0.7)
plt.title("Montant moyen des commandes selon les frais de livraison")
plt.xlabel("Frais de livraison (DH)")
plt.ylabel("Montant moyen (DH)")

for i, montant in enumerate(montant_moyen_livraison.values):
    plt.text(i, montant + 100, f"{montant:,.0f} DH", ha="center", va="bottom")

plt.tight_layout()
plt.show()


In [ ]:
#df_valide qui conserve toutes les commandes dont le statut est différent de "cancelled"

df_valid=orders_sample[
    orders_sample["status"]!="cancelled"
].copy()

print(df_valid.shape)


In [ ]:
#Calculer le montant moyen par tarif de livraison
montant_moyen_valid = (
    df_valid
    .groupby("shipping_fee")["total_amount"]
    .mean()
)

print(montant_moyen_valid.round(2))

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    montant_moyen_valid.index.astype(str),
    montant_moyen_valid.values,
    width=0.6
)
for i, montant in enumerate(montant_moyen_valid.values):
    plt.text(i, montant + 100, f"{montant:,.0f} DH", ha="center", va="bottom")

plt.title("Montant moyen des commandes par tarif de livraison")
plt.xlabel("Frais de livraison (DH)")
plt.ylabel("Montant moyen (DH)")

plt.tight_layout()
plt.show()


In [ ]:

# Calculer le chiffre d'affaires par statut
ca_par_statut = (
    df_valid
    .groupby("status")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

print(ca_par_statut.round(2))


In [ ]:
# Convertir le chiffre d'affaires en millions de DH
ca_millions = ca_par_statut / 1_000_000

plt.figure(figsize=(10, 5))

plt.bar(ca_millions.index, ca_millions.values)
# Afficher le chiffre d'affaires au-dessus de chaque barre
for i, valeur in enumerate(ca_millions.values):
    plt.text(
        i,
        valeur,
        f"{valeur:,.1f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.title("Chiffre d'affaires par statut de commande")
plt.xlabel("Statut de commande")
plt.ylabel("Chiffre d'affaires (millions de DH)")

plt.tight_layout()
plt.show()

#Nous excluons les commandes annulées afin de ne pas surestimer le chiffre d'affaires
#  associé aux commandes non annulées. Une commande 
# annulée ne représente pas nécessairement une vente réellement réalisée.

In [ ]:
#Quel pourcentage du chiffre d'affaires total non annulé est généré par chaque statut ?
part_ca_statut = (
    ca_par_statut / ca_par_statut.sum()
) * 100

print(part_ca_statut.round(2))

In [ ]:

plt.pie(
    part_ca_statut,
    labels=part_ca_statut.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Répartition du chiffre d'affaires par statut")
plt.tight_layout()
plt.savefig(
    dossier_graphiques / "Répartition du chiffre d'affaires par statut.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:

# Filtrer les commandes non annulées de 2025
df_2025 = df_valid[df_valid["annee"] == 2025].copy()

# Calculer le chiffre d'affaires par mois
ca_mensuel = (
    df_2025
    .groupby("mois")["total_amount"]
    .sum()
    .reindex(range(1, 13), fill_value=0)
)

print(ca_mensuel)


In [ ]:
plt.figure(figsize=(11, 5))

plt.plot(
    ca_mensuel.index,
    ca_mensuel.values,
    marker="o"
)

plt.title("Évolution mensuelle du chiffre d'affaires en 2025")
plt.xlabel("Mois")
plt.ylabel("Chiffre d'affaires (DH)")
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

# Calculer la variation du chiffre d'affaires par rapport au mois précédent
variation_ca = ca_mensuel.diff()

print(variation_ca.round(2))


In [ ]:

# Calculer la variation mensuelle en pourcentage
variation_pct = ca_mensuel.pct_change() * 100

print(variation_pct.round(2))


In [ ]:
#une courbe représentant les variations en pourcentage

plt.figure(figsize=(10, 5))

plt.plot(
    variation_pct.index,
    variation_pct.values,
    marker="o"
)

plt.axhline(y=0, linestyle="--" )

plt.title("Variation mensuelle du chiffre d'affaires en 2025")
plt.xlabel("Mois")
plt.ylabel("Variation (%)")
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

# Identifier le mois avec la plus forte baisse
mois_plus_forte_baisse = variation_pct.idxmin()
valeur_plus_forte_baisse = variation_pct.min()

print("Mois de la plus forte baisse :", mois_plus_forte_baisse)
print("Variation :", round(valeur_plus_forte_baisse, 2), "%")


In [ ]:
#Objectif métier : comprendre comment les montants
#  des commandes sont répartis et repérer
#  les commandes dont les montants sont exceptionnellement élevés ou faibles.


# Garder les commandes de 2025, sauf les commandes annulées
df_2025 = df_valid[df_valid["annee"] == 2025].copy()

# Créer la boîte à moustaches
plt.figure(figsize=(9, 5))
plt.boxplot(df_2025["total_amount"])

plt.title("Distribution des montants des commandes en 2025")
plt.ylabel("Montant de la commande (DH)")
plt.xticks([1], ["Commandes 2025"])
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

# Calculer les quartiles
Q1 = df_2025["total_amount"].quantile(0.25)
Q3 = df_2025["total_amount"].quantile(0.75)

# Calculer l'écart interquartile
IQR = Q3 - Q1

# Définir la limite supérieure des valeurs atypiques
limite_sup = Q3 + 1.5 * IQR

# Compter les commandes atypiques
nb_outliers = (df_2025["total_amount"] > limite_sup).sum()

print("Q1 :", round(Q1, 2), "DH")
print("Q3 :", round(Q3, 2), "DH")
print("Limite supérieure :", round(limite_sup, 2), "DH")
print("Nombre de valeurs atypiques :", nb_outliers)


In [ ]:
#---------------------------------------KPI--------------------------------------#

In [ ]:
#Question métier : quel est le chiffre d'affaires total des commandes non annulées ?


# Calculer le chiffre d'affaires total
chiffre_affaires = df_valid["total_amount"].sum()

print(f"Chiffre d'affaires : {chiffre_affaires:,.2f} DH")

In [ ]:
#Question métier : combien de commandes ne sont pas annulées dans notre 
# base e-commerce ?
# Compter les commandes non annulées
nombre_commandes = df_valid["order_id"].nunique()

print(f"Nombre de commandes valides : {nombre_commandes:,}")

In [ ]:
#Question métier : en moyenne, quel montant représente une commande non annulée ?


# Calculer le panier moyen
panier_moyen = (
    df_valid["total_amount"].sum()
    / df_valid["order_id"].nunique()
)

print(f"Panier moyen : {panier_moyen:,.2f} DH")


In [ ]:
#Question métier : quel pourcentage des commandes passées a été annulé ?

# Compter toutes les commandes distinctes
total_commandes = orders_sample["order_id"].nunique()

# Compter les commandes annulées
commandes_annulees = orders_sample.loc[
    orders_sample["status"] == "cancelled",
    "order_id"
].nunique()

# Calculer le taux d'annulation
taux_annulation = (
    commandes_annulees / total_commandes
) * 100

print(f"Total des commandes : {total_commandes:,}")
print(f"Commandes annulées : {commandes_annulees:,}")
print(f"Taux d'annulation : {taux_annulation:.2f} %")


In [ ]:
#Question métier : le chiffre d'affaires augmente-t-il ou diminue-t-il d'une
#  année à l'autre ?

# Calculer le chiffre d'affaires par année
ca_annuel = (
    df_valid.groupby("annee")["total_amount"]
    .sum()
)

# Calculer la variation en pourcentage d'une année à l'autre
variation_annuelle = ca_annuel.pct_change() * 100

print("Chiffre d'affaires par année :")
print(ca_annuel.round(2))

print("\nVariation annuelle (%) :")
print(variation_annuelle.round(2))


In [ ]:
#Calculer la variation du chiffre d'affaires entre 2023 et 2025

ca_2023 = ca_annuel.loc[2023]
ca_2025 = ca_annuel.loc[2025]

variation_totale = (
    (ca_2025 - ca_2023) / ca_2023
) * 100

print(f"Variation du CA entre 2023 et 2025 : {variation_totale:.2f} %")


In [ ]:

# Regrouper les commandes de 2025 par mois
kpi_mensuels = (
    df_2025.groupby("mois")
    .agg(
        chiffre_affaires=("total_amount", "sum"),
        nombre_commandes=("order_id", "nunique")
    )
    .reindex(range(1, 13), fill_value=0)
)

print(kpi_mensuels.round(2))


In [ ]:
#Calculer le panier moyen par mois


# Ajouter le panier moyen au tableau mensuel
kpi_mensuels["panier_moyen"] = (
    kpi_mensuels["chiffre_affaires"]
    / kpi_mensuels["nombre_commandes"]
)

print(kpi_mensuels.round(2))


In [ ]:

mois_meilleur_panier = kpi_mensuels["panier_moyen"].idxmax()
meilleur_panier = kpi_mensuels["panier_moyen"].max()

print("Mois du meilleur panier moyen :", mois_meilleur_panier)
print(f"Panier moyen : {meilleur_panier:,.2f} DH")


In [ ]:

# Compter les commandes et calculer leur montant par statut
kpi_statuts = (
    orders_sample.groupby("status")
    .agg(
        nombre_commandes=("order_id", "nunique"),
        montant_total=("total_amount", "sum")
    )
    .sort_values("nombre_commandes", ascending=False)
)

print(kpi_statuts.round(2))


In [ ]:
# Nombre total de commandes
total_commandes = orders_sample["order_id"].nunique()

# Pourcentage de commandes par statut
kpi_statuts["pourcentage_commandes"] = (
    kpi_statuts["nombre_commandes"] / total_commandes
) * 100

print(kpi_statuts.round(2))

In [ ]:
total_pourcentage = kpi_statuts["pourcentage_commandes"].sum()

print(f"Total des pourcentages : {total_pourcentage:.2f} %")

In [ ]:
kpi_globaux = pd.DataFrame({
    "indicateur": [
        "Nombre total de commandes",
        "Commandes non annulées",
        "Chiffre d'affaires non annulé (DH)",
        "Panier moyen (DH)",
        "Taux d'annulation (%)"
    ],
    "valeur": [
        total_commandes,
        nombre_commandes,
        chiffre_affaires,
        panier_moyen,
        taux_annulation
    ]
})

print(kpi_globaux)

In [ ]:
kpi_globaux_affichage = kpi_globaux.copy()

kpi_globaux_affichage["valeur"] = [
    f"{total_commandes:,}".replace(",", " "),
    f"{nombre_commandes:,}".replace(",", " "),
    f"{chiffre_affaires:,.2f} DH",
    f"{panier_moyen:,.2f} DH",
    f"{taux_annulation:.2f} %"
]

print(kpi_globaux_affichage.to_string(index=False))

In [ ]:
print(kpi_mensuels.round(2).to_string())

In [ ]:
print("Total des commandes mensuelles :", kpi_mensuels["nombre_commandes"].sum())

print(
    "Chiffre d'affaires mensuel total :",
    round(kpi_mensuels["chiffre_affaires"].sum(), 2)
)

print(
    "Chiffre d'affaires annuel calculé :",
    round(ca_annuel.loc[2025], 2)
)

In [ ]:
kpi_annuels = pd.DataFrame({
    "chiffre_affaires": ca_annuel,
    "variation_annuelle_pct": variation_annuelle
})

print(kpi_annuels.round(2).to_string())

In [ ]:
print(kpi_statuts.columns.tolist())

In [ ]:
chemin_excel = "../reports/rapport_kpi_ecommerce.xlsx"

with pd.ExcelWriter(chemin_excel, engine="openpyxl") as writer:
    kpi_globaux.to_excel(
        writer, sheet_name="KPI_Globaux", index=False
    )
    kpi_mensuels.to_excel(
        writer, sheet_name="KPI_Mensuels_2025"
    )
    kpi_statuts.to_excel(
        writer, sheet_name="KPI_Statuts"
    )
    kpi_annuels.to_excel(
        writer, sheet_name="KPI_Annuels"
    )

print("Fichier Excel créé avec succès :", chemin_excel)

In [ ]:
%pip install openpyxl

In [ ]:
import openpyxl

print("Version installée :", openpyxl.__version__)

In [ ]:
from pathlib import Path

dossier_graphiques = Path("../reports/graphiques")
dossier_graphiques.mkdir(parents=True, exist_ok=True)

print("Dossier créé :", dossier_graphiques.resolve())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11, 5))
plt.plot(ca_mensuel.index, ca_mensuel.values, marker="o")

plt.title("Évolution mensuelle du chiffre d'affaires en 2025")
plt.xlabel("Mois")
plt.ylabel("Chiffre d'affaires (DH)")
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    dossier_graphiques / "01_ca_mensuel_2025.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

In [ ]:
#----------------------------Vérification avant modification--------------------#
from pathlib import Path

chemin_excel = Path("../reports/rapport_kpi_ecommerce.xlsx")

print("Fichier trouvé :", chemin_excel.exists())
print("Chemin :", chemin_excel.resolve())

In [ ]:

from openpyxl import load_workbook
from pathlib import Path

# Chemin du fichier Excel
chemin_excel = Path("../reports/rapport_kpi_ecommerce.xlsx")

# Ouvrir le classeur existant
classeur = load_workbook(chemin_excel)

# Créer Dashboard uniquement si elle n'existe pas déjà
if "Dashboard" not in classeur.sheetnames:
    dashboard = classeur.create_sheet("Dashboard")
else:
    dashboard = classeur["Dashboard"]

# Ajouter le titre
dashboard["A1"] = "DASHBOARD E-COMMERCE — 2025"

# Enregistrer le fichier
classeur.save(chemin_excel)
classeur.close()

print("Feuille Dashboard créée avec succès !")


In [ ]:
classeur = load_workbook(chemin_excel, read_only=True)
print(classeur.sheetnames)
classeur.close()

In [ ]:

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from pathlib import Path

chemin_excel = Path("../reports/rapport_kpi_ecommerce.xlsx")

classeur = load_workbook(chemin_excel)
dashboard = classeur["Dashboard"]

# Titre principal
dashboard.merge_cells("A1:H1")
dashboard["A1"] = "DASHBOARD E-COMMERCE — 2025"
dashboard["A1"].font = Font(
    name="Arial", size=18, bold=True, color="FFFFFF"
)
dashboard["A1"].fill = PatternFill(
    "solid", fgColor="17365D"
)
dashboard["A1"].alignment = Alignment(
    horizontal="center", vertical="center"
)
dashboard.row_dimensions[1].height = 35

# Titres des KPI
kpis = {
    "A3": ("Chiffre d'affaires (DH)", 5109417459.57),
    "C3": ("Commandes non annulées", 189885),
    "E3": ("Panier moyen (DH)", 26907.96),
    "G3": ("Taux d'annulation", 0.050575)
}

for cellule, (titre, valeur) in kpis.items():
    col = dashboard[cellule].column
    ligne_valeur = 4

    dashboard[cellule] = titre
    dashboard[cellule].font = Font(bold=True, color="FFFFFF")
    dashboard[cellule].fill = PatternFill(
        "solid", fgColor="4472C4"
    )
    dashboard[cellule].alignment = Alignment(
        horizontal="center", vertical="center", wrap_text=True
    )

    valeur_cellule = dashboard.cell(row=ligne_valeur, column=col)
    valeur_cellule.value = valeur
    valeur_cellule.font = Font(bold=True, size=14, color="17365D")
    valeur_cellule.alignment = Alignment(horizontal="center")

    # Format des nombres
    if "Taux" in titre:
        valeur_cellule.number_format = "0.00%"
    elif "Commandes" in titre:
        valeur_cellule.number_format = '#,##0'
    else:
        valeur_cellule.number_format = '#,##0.00'

    dashboard.column_dimensions[valeur_cellule.column_letter].width = 24

# Bordure légère autour des indicateurs
bordure = Border(
    bottom=Side(style="thin", color="D9E2F3")
)

for ligne in dashboard.iter_rows(min_row=3, max_row=4, min_col=1, max_col=8):
    for cellule in ligne:
        cellule.border = bordure

dashboard.row_dimensions[3].height = 35
dashboard.row_dimensions[4].height = 35

classeur.save(chemin_excel)
classeur.close()

print("Les 4 KPI ont été ajoutés au Dashboard !")


In [ ]:

from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from pathlib import Path

# Chemins des fichiers
chemin_excel = Path("../reports/rapport_kpi_ecommerce.xlsx")
chemin_image = dossier_graphiques / "01_ca_mensuel_2025.png"

# Vérifier que l'image existe
if not chemin_image.exists():
    raise FileNotFoundError(f"Image introuvable : {chemin_image}")

# Ouvrir le classeur existant
classeur = load_workbook(chemin_excel)
dashboard = classeur["Dashboard"]

# Ajouter le graphique à partir de l'image PNG
graphique = Image(str(chemin_image))

# Ajuster les dimensions pour le Dashboard
graphique.width = 720
graphique.height = 360

# Positionner l'image à partir de la cellule A7
dashboard.add_image(graphique, "A7")

# Enregistrer le classeur
classeur.save(chemin_excel)
classeur.close()

print("Graphique mensuel ajouté au Dashboard !")


In [ ]:

from pathlib import Path

for fichier in sorted(dossier_graphiques.glob("*.png")):
    print(fichier.name)


In [ ]:

from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from pathlib import Path

chemin_excel = Path("../reports/rapport_kpi_ecommerce.xlsx")

# Graphiques et cellules de destination
graphiques = [
    ("CA_Par_Status.png", "A27"),
    ("Ca_Par_Region.png", "A47"),
    ("Nombre_de_commandes_par_région.png", "J7"),
]

classeur = load_workbook(chemin_excel)
dashboard = classeur["Dashboard"]

# Éviter de réinsérer les mêmes images si le code est relancé
for nom_fichier, cellule in graphiques:
    chemin_image = dossier_graphiques / nom_fichier

    if not chemin_image.exists():
        print(f"Image introuvable : {nom_fichier}")
        continue

    image = Image(str(chemin_image))
    image.width = 600
    image.height = 320
    image.anchor = cellule
    dashboard.add_image(image)

    print(f"Ajouté : {nom_fichier} en {cellule}")

classeur.save(chemin_excel)
classeur.close()

print("Dashboard mis à jour !")


In [ ]:
%pip install reportlab

In [ ]:

try:
    import reportlab
    print("ReportLab est installé :", reportlab.Version)
except ImportError:
    print("ReportLab n'est pas encore installé.")


In [ ]:

from pathlib import Path

dossier_graphiques = Path(
    r"C:\Users\ELHADDOUCHI Asmae\Documents\ecommerce_data_analytics\reports\graphiques"
)

graphiques_rapport = [
    "01_ca_mensuel_2025.png",
    "CA_Par_Status.png",
    "Ca_Par_Region.png",
    "Nombre_de_commandes_par_région.png",
]

for nom in graphiques_rapport:
    chemin = dossier_graphiques / nom
    print(f"{'OK' if chemin.exists() else 'MANQUANT'} : {nom}")


In [ ]:

from pathlib import Path

from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image,
    Table, TableStyle, PageBreak
)

# --------------------------------------------------
# 1. Chemins des fichiers
# --------------------------------------------------
dossier_projet = Path.cwd().parent
dossier_reports = dossier_projet / "reports"
dossier_graphiques = dossier_reports / "graphiques"

dossier_reports.mkdir(parents=True, exist_ok=True)

chemin_pdf = dossier_reports / "rapport_analyse_ecommerce.pdf"

# --------------------------------------------------
# 2. Configuration du document
# --------------------------------------------------
document = SimpleDocTemplate(
    str(chemin_pdf),
    pagesize=A4,
    rightMargin=1.8 * cm,
    leftMargin=1.8 * cm,
    topMargin=1.8 * cm,
    bottomMargin=1.8 * cm
)

styles = getSampleStyleSheet()

styles.add(ParagraphStyle(
    name="TitreProjet",
    parent=styles["Title"],
    fontName="Helvetica-Bold",
    fontSize=23,
    leading=29,
    alignment=TA_CENTER,
    textColor=colors.HexColor("#17365D"),
    spaceAfter=18
))

styles.add(ParagraphStyle(
    name="SousTitre",
    parent=styles["Normal"],
    fontSize=12,
    leading=18,
    alignment=TA_CENTER,
    textColor=colors.HexColor("#4472C4"),
    spaceAfter=15
))

styles.add(ParagraphStyle(
    name="Section",
    parent=styles["Heading1"],
    fontSize=16,
    leading=20,
    textColor=colors.HexColor("#17365D"),
    spaceBefore=12,
    spaceAfter=10
))

styles.add(ParagraphStyle(
    name="SousSection",
    parent=styles["Heading2"],
    fontSize=12,
    leading=16,
    textColor=colors.HexColor("#4472C4"),
    spaceBefore=8,
    spaceAfter=6
))

styles.add(ParagraphStyle(
    name="TexteProjet",
    parent=styles["BodyText"],
    fontSize=9.5,
    leading=14,
    spaceAfter=7
))

elements = []

def ajouter_texte(texte, style="TexteProjet"):
    elements.append(Paragraph(texte, styles[style]))

def ajouter_graphique(nom_fichier, largeur_max=16.5*cm,
                      hauteur_max=8.5*cm):
    chemin = dossier_graphiques / nom_fichier

    if not chemin.exists():
        ajouter_texte(
            f"Graphique indisponible : {nom_fichier}"
        )
        return

    image = Image(str(chemin))
    ratio = min(
        largeur_max / image.imageWidth,
        hauteur_max / image.imageHeight,
        1
    )
    image.drawWidth = image.imageWidth * ratio
    image.drawHeight = image.imageHeight * ratio
    elements.append(image)
    elements.append(Spacer(1, 0.3 * cm))

# --------------------------------------------------
# 3. Page de couverture
# --------------------------------------------------
elements.append(Spacer(1, 3 * cm))
ajouter_texte("ANALYSE DES VENTES<br/>E-COMMERCE",
              "TitreProjet")
ajouter_texte(
    "Projet Data Analytics | Période étudiée : 2023–2025",
    "SousTitre"
)
elements.append(Spacer(1, 1 * cm))

ajouter_texte(
    "Objectif : analyser les performances commerciales, "
    "l'évolution du chiffre d'affaires, les statuts des commandes "
    "et la répartition géographique des ventes."
)

elements.append(Spacer(1, 1 * cm))
ajouter_texte(
    "<b>Technologies utilisées</b><br/>"
    "SQL / MariaDB, Python, pandas, Matplotlib, Excel et ReportLab."
)
elements.append(Spacer(1, 2 * cm))
ajouter_texte(
    "Portfolio Junior Data Analyst",
    "SousTitre"
)
elements.append(PageBreak())

# --------------------------------------------------
# 4. Objectifs et méthodologie
# --------------------------------------------------
ajouter_texte("1. Contexte et objectifs", "Section")

ajouter_texte(
    "Ce projet consiste à exploiter un jeu de données e-commerce "
    "contenant 200 000 commandes. L'analyse vise à transformer les "
    "données en indicateurs utiles au suivi de l'activité."
)

ajouter_texte("Questions métier", "SousSection")

for question in [
    "Comment évolue le chiffre d'affaires au fil des mois et des années ?",
    "Quelle est la répartition des commandes selon leur statut ?",
    "Quelles régions présentent le plus grand volume de commandes "
    "et de chiffre d'affaires ?",
    "Quel est le montant moyen d'une commande non annulée ?",
    "Quelle proportion des commandes est annulée ?"
]:
    ajouter_texte("• " + question)

ajouter_texte("2. Méthodologie", "Section")
ajouter_texte(
    "<b>1.</b> Extraction et exploration des données issues de MariaDB.<br/>"
    "<b>2.</b> Préparation des données avec Python et pandas.<br/>"
    "<b>3.</b> Calcul des KPI commerciaux et analyse des tendances.<br/>"
    "<b>4.</b> Création des graphiques avec Matplotlib.<br/>"
    "<b>5.</b> Présentation des résultats dans Excel et ce rapport PDF."
)

ajouter_texte(
    "Pour le calcul du chiffre d'affaires présenté ci-dessous, "
    "les commandes au statut « cancelled » sont exclues. "
    "Le nombre total de commandes inclut, lui, tous les statuts."
)

# --------------------------------------------------
# 5. KPI principaux
# --------------------------------------------------
ajouter_texte("3. Indicateurs clés de performance (KPI)", "Section")

donnees_kpi = [
    ["Indicateur", "Résultat"],
    ["Nombre total de commandes", "200 000"],
    ["Commandes non annulées", "189 885"],
    ["Chiffre d'affaires hors annulations", "5 109 417 459,57 DH"],
    ["Panier moyen hors annulations", "26 907,96 DH"],
    ["Taux d'annulation", "5,06 %"]
]

table_kpi = Table(
    donnees_kpi,
    colWidths=[9.2 * cm, 7 * cm],
    repeatRows=1
)
table_kpi.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#17365D")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("FONTNAME", (0, 1), (-1, -1), "Helvetica"),
    ("FONTSIZE", (0, 0), (-1, -1), 9),
    ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#D9E2F3")),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1),
     [colors.white, colors.HexColor("#F2F6FC")]),
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ("TOPPADDING", (0, 0), (-1, -1), 8),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
]))
elements.append(table_kpi)
elements.append(Spacer(1, 0.4 * cm))

ajouter_texte(
    "Le panier moyen correspond au chiffre d'affaires hors annulations "
    "divisé par le nombre de commandes non annulées."
)

# --------------------------------------------------
# 6. Analyse visuelle
# --------------------------------------------------
elements.append(PageBreak())
ajouter_texte("4. Analyse des graphiques", "Section")

ajouter_texte("4.1. Évolution mensuelle des ventes en 2025",
              "SousSection")
ajouter_graphique("01_ca_mensuel_2025.png")
ajouter_texte(
    "Le mois de mars présente le chiffre d'affaires mensuel le plus "
    "élevé (environ 147,68 millions DH), tandis que février présente "
    "le plus faible (environ 133,24 millions DH). Ces variations "
    "méritent d'être examinées avec des données complémentaires "
    "sur les produits, les clients et les campagnes commerciales."
)

ajouter_texte("4.2. Chiffre d'affaires par statut", "SousSection")
ajouter_graphique("CA_Par_Status.png")
ajouter_texte(
    "Les montants par statut permettent de distinguer les commandes "
    "terminées, livrées, expédiées, en cours de traitement et annulées. "
    "Il faut interpréter ces catégories selon les règles métier : "
    "un montant associé à un statut ne constitue pas nécessairement "
    "un chiffre d'affaires comptable reconnu."
)

ajouter_texte("4.3. Chiffre d'affaires par région", "SousSection")
ajouter_graphique("Ca_Par_Region.png")

elements.append(PageBreak())
ajouter_texte("4.4. Nombre de commandes par région", "SousSection")
ajouter_graphique("Nombre_de_commandes_par_région.png")

ajouter_texte("5. Analyse annuelle", "Section")

donnees_annuelles = [
    ["Année", "Chiffre d'affaires hors annulations"],
    ["2023", "1 707 172 840,87 DH"],
    ["2024", "1 701 507 094,78 DH"],
    ["2025", "1 700 737 523,92 DH"]
]
table_annuelle = Table(
    donnees_annuelles,
    colWidths=[4 * cm, 12.2 * cm],
    repeatRows=1
)
table_annuelle.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#17365D")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("FONTSIZE", (0, 0), (-1, -1), 9),
    ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#D9E2F3")),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1),
     [colors.white, colors.HexColor("#F2F6FC")]),
    ("TOPPADDING", (0, 0), (-1, -1), 8),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
]))
elements.append(table_annuelle)
elements.append(Spacer(1, 0.3 * cm))

ajouter_texte(
    "Le chiffre d'affaires annuel diminue légèrement sur la période : "
    "la variation entre 2023 et 2025 est d'environ -0,38 %. "
    "Cette stabilité relative invite à approfondir l'analyse des "
    "produits, des clients, des prix et des volumes."
)

# --------------------------------------------------
# 7. Recommandations et conclusion
# --------------------------------------------------
ajouter_texte("6. Recommandations métier", "Section")

for recommandation in [
    "Analyser les raisons des annulations par région, statut, période "
    "et type de produit.",
    "Examiner les mois les plus performants et les moins performants "
    "afin d'identifier des facteurs possibles, sans conclure à une "
    "causalité sans données supplémentaires.",
    "Comparer le chiffre d'affaires au volume de commandes et au "
    "panier moyen pour mieux comprendre la performance commerciale.",
    "Mettre en place un suivi régulier des KPI dans un dashboard "
    "actualisé à partir de la base de données."
]:
    ajouter_texte("• " + recommandation)

ajouter_texte("7. Conclusion", "Section")
ajouter_texte(
    "Ce projet met en pratique les principales étapes d'un travail "
    "de Data Analyst : extraction SQL, exploration, préparation avec "
    "pandas, calcul des indicateurs, visualisation et communication "
    "des résultats. Les analyses présentées constituent une base "
    "pour des investigations plus détaillées sur les produits, "
    "les clients et les causes des annulations."
)

ajouter_texte("Compétences mobilisées", "SousSection")
ajouter_texte(
    "SQL • Python • pandas • Matplotlib • Excel • KPI e-commerce • "
    "Analyse exploratoire • Reporting"
)

# --------------------------------------------------
# 8. Pied de page et génération du PDF
# --------------------------------------------------
def pied_de_page(canvas, doc):
    canvas.saveState()
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(colors.grey)
    canvas.drawString(1.8 * cm, 1 * cm, "Portfolio Junior Data Analyst")
    canvas.drawRightString(
        A4[0] - 1.8 * cm, 1 * cm, f"Page {doc.page}"
    )
    canvas.restoreState()

document.build(
    elements,
    onFirstPage=pied_de_page,
    onLaterPages=pied_de_page
)

print("PDF créé avec succès :", chemin_pdf.resolve())
print("Taille du fichier :", round(chemin_pdf.stat().st_size / 1024), "Ko")
